<div style="background-color:#BBD7FF;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">



# ⚛️ Wprowadzenie do obliczeń kwantowych
## 🧮 Laboratorium 5 — część III: zadania domowe ze wskazówkami
### **Katedra Informatyki, Mechaniki i Obliczeń Kwantowych**

---

Notebook zawiera dwa zadania z **Części IV: Zadania do samodzielnej realizacji** dokumentu do laboratorium 5: wieloargumentową koniunkcję i pełny sumator dla liczb o zmiennej długości bitowej.

Przy każdym zadaniu znajdują się wskazówki oraz miejsce na obliczenia, kod i wnioski. Przykład poprzedzający zadania pokazuje kompletny sposób opracowania odpowiedzi na innym problemie.

</div>

<div style="background-color:#f8f8f8;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## 💡 Jak przygotować i oddać rozwiązanie

---

Dla **każdego z dwóch zadań** przygotuj trzy części:

1. **Analiza:** opisz działanie algorytmu przed napisaniem `QuantumCircuit`. Podaj mapowanie kubitów, stan początkowy, równania logiczne i przewidywany wynik. Wyznacz zapotrzebowanie na kubity. Dla zadania 2 prześledź przeniesienia na przykładzie $5+3$.
2. **Weryfikacja Python/Qiskit:** zbuduj obwód, sprawdź wybrane pełne stany przez `Statevector`, a następnie wyniki pomiarów w `AerSimulator`. Przewidywanie ma pochodzić z własnego rachunku lub niezależnego wzorca klasycznego.
3. **Wniosek:** opisz zgodność wyników, obsługę przeniesienia lub kubitów roboczych, ograniczenia zasobów i ewentualne poprawki.

Funkcje `q_and(*args)` i `quantum_add(a, b)` mają zwracać `int`. Funkcja `build_add_circuit(a, b)` ma budować i zwracać obwód `QuantumCircuit` z pomiarami. Klasyczne `all(args)` i `a + b` służą wyłącznie jako wzorce testów, a nie jako implementacje tych funkcji.

Obliczenia wpisuj w komórkach Markdown, a programy w komórkach kodu. Przed oddaniem uruchom notebook od początku w świeżym kernelu i zapisz go z wynikami. Komórki studenckie są początkowo komentarzami, dlatego gotowy szablon można wykonać bez uzupełniania rozwiązań.

</div>

<div style="background-color:#f8f8f8;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## 🔑 Karta referencyjna: rejestry i kontrola wyników

---

| Pojęcie | Konwencja lub narzędzie | Na co uważać |
|---|---|---|
| Kolejność kubitów | $\lvert q_{N-1}\ldots q_1q_0\rangle$ | `q0` jest najmniej znaczącym bitem. |
| Przygotowanie liczby | Bit $i$: `(a >> i) & 1` | Bit $i$ umieszczaj na kubicie $A_i$, niezależnie od jego globalnego indeksu. |
| Wynik pomiaru | Klucz `counts`: $c_{m-1}\ldots c_0$ | Decyduje mapowanie kubitów do bitów klasycznych. |
| Szerokość wejść sumatora | `n = max(1, a.bit_length(), b.bit_length())` | Dla zera `bit_length()` wynosi 0. |
| Szerokość sumy | $n+1$ bitów | Nie wolno odrzucić końcowego przeniesienia. |
| Symulacja stanu | `Statevector.from_instruction(qc)` | Obwód bez pomiarów; stan oczekiwany oblicz niezależnie. |
| Pomiary | `uruchom_pomiary(qc, shots=256)` | Obwód musi zawierać jawne pomiary do bitów klasycznych. |
| Odwracalność | $\lvert x,t\rangle\mapsto\lvert x,t\oplus f(x)\rangle$ | Wymaga zachowania informacji, a nie tylko poprawnego pojedynczego bitu. |
| Uncomputacja | Odwrócenie operacji przygotowujących wyniki robocze | Nie usuń przy tym wyniku końcowego. |

Idealny Aer nie zawiera domyślnie modelu szumu. Dla obwodów z X, CNOT, CCX i MCX oraz wejść bazowych poprawny wynik jest deterministyczny. Wybór najczęstszego klucza nie powinien maskować błędnego rozkładu: sprawdzaj, czy wszystkie zliczenia przypadły oczekiwanemu wynikowi.

Pełny wektor stanu zawiera $2^N$ amplitud. Dla typu `complex128` sam wektor zajmuje około $16\cdot2^N$ bajtów, a symulacja wymaga dodatkowej pamięci. Testy zaczynaj od małych wejść; w obu funkcjach dodaj jawny limit zasobów i czytelny błąd przed alokacją zbyt dużego obwodu.

</div>

<div style="background-color:#f8f8f8;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## Przygotowanie środowiska

---

Potrzebne są `numpy`, `qiskit`, `qiskit-aer` i Jupyter/IPython. Jeśli pakietów brakuje, odkomentuj polecenie instalacji. Po instalacji w razie potrzeby uruchom ponownie kernel.

</div>

In [1]:
# Opcjonalna instalacja w środowisku aktywnego kernela:
# %pip install numpy qiskit qiskit-aer ipython

<div style="background-color:#f8f8f8;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## 🛠️ Uniwersalne funkcje pomocnicze

---

Funkcje są osadzone w notebooku; nie trzeba importować osobnego pliku.

| Funkcja | Zastosowanie |
|---|---|
| `prepare_basis_state(label)` | Przygotowanie bazowego stanu z etykiety binarnej. |
| `pokaz_stan(state, nazwa)` | Stan w notacji Diraca, np. `pokaz_stan(state, r"\psi")`. |
| `pokaz_prawdopodobienstwa(state, kubity=[0])` | Tabela Markdown dokładnych prawdopodobieństw, także dla podrejestru. |
| `pokaz_macierz(matrix, nazwa)` | Macierz w LaTeX. |
| `pokaz_tabele(naglowki, rows)` | Tabela wyników i testów. |
| `check_state(state, expected)` | Zgodność pełnych amplitud i normalizacja. |
| `odczytaj_pewny_bit(state, kubit)` | Odczyt bitu o deterministycznym wyniku w bazie obliczeniowej. |
| `uruchom_pomiary(qc, shots=256)` | Uruchomienie idealnej symulacji Aer; zwraca `counts`. |
| `pokaz_zliczenia(counts)` | Zliczenia, częstości i procenty. |
| `najczestszy_bit(counts)` | Odczyt wyniku dla **jednego** bitu klasycznego; odrzuca remis. |
| `bit(value)` | Walidacja argumentu równego 0 albo 1. |

`pokaz_prawdopodobienstwa` nie wykonuje pomiarów. Dla obwodu zakłada wejście zerowe i wymaga braku pomiarów. W zadaniu 2 nie używaj `najczestszy_bit` do odczytu całego wyniku wielobitowego.

</div>

In [1]:
import html
import numpy as np
import qiskit
from IPython.display import HTML, Math, Markdown, display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import array_to_latex

np.set_printoptions(precision=6, suppress=True)
ATOL = 1e-10


def prepare_basis_state(label):
    """Przygotuj |q_(n-1)...q_0>; prawy znak etykiety odpowiada q0."""
    if not label or any(bit not in '01' for bit in label):
        raise ValueError("Podaj niepustą etykietę binarną.")
    qc = QuantumCircuit(len(label))
    for qubit, bit in enumerate(reversed(label)):
        if bit == '1':
            qc.x(qubit)
    return qc


def check_state(state, expected):
    """Sprawdź normalizację oraz zgodność wszystkich amplitud, łącznie z fazą."""
    expected = np.asarray(expected, dtype=complex)
    assert np.isclose(np.linalg.norm(state.data), 1, atol=ATOL, rtol=0)
    assert np.allclose(state.data, expected, atol=ATOL, rtol=0)


def _jako_stan(stan):
    """Wspólna konwersja danych używana przez funkcje prezentacji."""
    if isinstance(stan, QuantumCircuit):
        stan = Statevector.from_instruction(stan)
    elif not isinstance(stan, Statevector):
        stan = Statevector(stan)
    if stan.num_qubits is None:
        raise ValueError("Podaj stan rejestru kubitów (2, 4, 8, ... amplitud).")
    return stan


def pokaz_stan(stan, nazwa=r"\psi", precyzja=10):
    """Wyświetl stan w notacji Diraca; przyjmuje wektor lub obwód bez pomiarów."""
    stan = _jako_stan(stan)
    prefix = rf"\lvert {nazwa}\rangle = " if nazwa else ""
    latex = stan.draw(
        "latex_source", convention="ket", prefix=prefix,
        decimals=precyzja, max_size=stan.dim,
    )
    display(Math(latex))


def pokaz_prawdopodobienstwa(stan, precyzja=3, *, kubity=None):
    """Wyświetl prawdopodobieństwa w bazie obliczeniowej jako tabelę Markdown.

    Przyjmuje Statevector, tablicę amplitud lub obwód bez pomiarów.
    kubity=None oznacza cały rejestr; kubity=[0] daje rozkład marginalny q0.
    Wiersze pochodzą bezpośrednio z probabilities_dict(), zgodnie z Qiskitem.
    """
    stan = _jako_stan(stan)
    if kubity is not None:
        kubity = list(kubity)
        if not kubity or len(set(kubity)) != len(kubity):
            raise ValueError("Lista kubitów musi być niepusta i bez powtórzeń.")
        if any(not isinstance(q, int) or not 0 <= q < stan.num_qubits for q in kubity):
            raise ValueError("Indeks kubitu wykracza poza rejestr lub nie jest całkowity.")

    state_header = "Stan"
    if kubity is not None:
        order = ', '.join(f'q{q}' for q in reversed(kubity))
        state_header += f" ({order})"
    rows = [
        f"| {state_header} | Prawdopodobieństwo | Procent |",
        "|:---:|---:|---:|",
    ]
    for label, probability in stan.probabilities_dict(qargs=kubity).items():
        rows.append(
            rf"| $\vert {label}\rangle$ "
            f"| {probability:.{precyzja}f} "
            f"| {100 * probability:.1f}% |"
        )
    display(Markdown("\n".join(rows)))


def pokaz_macierz(macierz, nazwa="M", precyzja=10):
    """Wyświetl macierz NumPy lub operator Qiskita jako macierz LaTeX."""
    if isinstance(macierz, Operator):
        macierz = macierz.data
    macierz = np.asarray(macierz, dtype=complex)
    if macierz.ndim != 2:
        raise ValueError("Oczekiwana jest tablica dwuwymiarowa.")
    prefix = rf"{nazwa} = " if nazwa else ""
    latex = array_to_latex(
        macierz, precision=precyzja, prefix=prefix, source=True,
        max_size=max(3, *macierz.shape),
    )
    display(Math(latex))


def pokaz_tabele(naglowki, wiersze):
    """Wyświetl czytelną tabelę HTML bez dodatkowej biblioteki tabelarycznej."""
    style = "padding:7px 12px;border-bottom:1px solid #cbd5e1;text-align:center;"
    heading = ''.join(f'<th style="{style}">{html.escape(str(v))}</th>' for v in naglowki)
    body = []
    for row in wiersze:
        if len(row) != len(naglowki):
            raise ValueError("Liczba komórek wiersza musi odpowiadać liczbie nagłówków.")
        body.append('<tr>' + ''.join(
            f'<td style="{style}">{html.escape(str(v))}</td>' for v in row
        ) + '</tr>')
    display(HTML(
        '<div style="overflow-x:auto;"><table style="border-collapse:collapse;">'
        f'<thead><tr>{heading}</tr></thead><tbody>{"".join(body)}</tbody></table></div>'
    ))



from itertools import product
from qiskit import transpile
from qiskit_aer import AerSimulator

backend = AerSimulator(max_parallel_threads=1)
wejscia2 = list(product((0, 1), repeat=2))


def bit(value):
    """Sprawdź klasyczne wejście 0/1, zamiast niejawnie je zaokrąglać."""
    if not isinstance(value, (int, np.integer, bool)) or value not in (0, 1):
        raise ValueError("Oczekiwano bitu 0 albo 1.")
    return int(value)


def odczytaj_pewny_bit(stan, kubit):
    """Zwróć bit, jeśli rozkład marginalny jest deterministyczny."""
    stan = _jako_stan(stan)
    if not isinstance(kubit, int) or not 0 <= kubit < stan.num_qubits:
        raise ValueError("Niepoprawny indeks kubitu.")
    p = stan.probabilities(qargs=[kubit])
    wynik = int(np.argmax(p))
    if not np.isclose(p[wynik], 1.0, atol=ATOL, rtol=0):
        raise ValueError("Ten kubit nie ma pewnego wyniku pomiaru w bazie obliczeniowej.")
    return wynik


def uruchom_pomiary(qc, shots=1024, seed=2026):
    """Idealne pomiary Aer. Obwód musi już zawierać mapowanie do bitów klasycznych."""
    if not isinstance(shots, int) or shots <= 0:
        raise ValueError("shots musi być dodatnią liczbą całkowitą.")
    if not qc.num_clbits or not any(inst.operation.name == 'measure' for inst in qc.data):
        raise ValueError("Najpierw dodaj pomiary do obwodu.")
    compiled = transpile(qc, backend, seed_transpiler=seed)
    return backend.run(compiled, shots=shots, seed_simulator=seed).result().get_counts()


def pokaz_zliczenia(counts):
    total = sum(counts.values())
    if total <= 0:
        raise ValueError("Brak zliczeń.")
    pokaz_tabele(["Wynik klasyczny", "Zliczenia", "Częstość", "Procent"], [
        [label, n, f"{n / total:.3f}", f"{100 * n / total:.1f}%"]
        for label, n in sorted(counts.items())
    ])


def najczestszy_bit(counts):
    """Zwróć dominujący bit 0/1; remis nie jest jednoznacznym rozstrzygnięciem."""
    if not counts or any(k not in ('0', '1') for k in counts):
        raise ValueError("Oczekiwano wyników pomiaru dokładnie jednego bitu.")
    maximum = max(counts.values())
    winners = [k for k, v in counts.items() if v == maximum]
    if maximum <= 0 or len(winners) != 1:
        raise ValueError("Brak jednoznacznego najczęstszego wyniku.")
    return int(winners[0])

print("Qiskit:", qiskit.__version__)
print("Gotowe: funkcje prezentacji, testowania i symulator Aer.")

Qiskit: 2.6.1
Gotowe: funkcje prezentacji, testowania i symulator Aer.


<div style="background-color:#e9f8e9;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## Przykład sposobu opracowania odpowiedzi — negacja bitu i pomiar

---

**Polecenie przykładowe:** dla wejścia $\lvert0\rangle$ zastosuj X, wyznacz stan i przewidź wynik 64 pomiarów w bazie obliczeniowej.

### A. Obliczenia analityczne

W bazie $(\lvert0\rangle,\lvert1\rangle)$:

$$X\lvert0\rangle=
\begin{bmatrix}0&1\\1&0\end{bmatrix}
\begin{bmatrix}1\\0\end{bmatrix}
=\begin{bmatrix}0\\1\end{bmatrix}=\lvert1\rangle.$$

Z reguły Borna $P(0)=0$, $P(1)=1$. Mapowanie $q_0\to c_0$ daje oczekiwane zliczenia `{'1': 64}`. Oczekiwany wektor zapisujemy przed wykonaniem symulacji.

### B. Weryfikacja Python/Qiskit

</div>

In [ ]:
expected_demo = np.array([0, 1], dtype=complex)
qc_demo = QuantumCircuit(1)
qc_demo.x(0)
state_demo = Statevector.from_instruction(qc_demo)
pokaz_stan(expected_demo, r"\psi_{\mathrm{teoria}}")
pokaz_stan(state_demo, r"\psi_{\mathrm{sym}}")
pokaz_prawdopodobienstwa(state_demo)
check_state(state_demo, expected_demo)

qc_demo_measure = QuantumCircuit(1, 1)
qc_demo_measure.compose(qc_demo, inplace=True)
qc_demo_measure.measure(0, 0)
counts_demo = uruchom_pomiary(qc_demo_measure, shots=64)
pokaz_zliczenia(counts_demo)
assert counts_demo == {'1': 64}
print(qc_demo_measure.draw(output="text"))

<div style="background-color:#feffe9;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

### C. Wniosek do przykładu

---

Bramka X zamieniła $\lvert0\rangle$ na $\lvert1\rangle$. Symulacja stanu potwierdziła amplitudy, a symulacja pomiarów — pewny wynik 1. Zgodność zliczeń wynika tutaj z deterministycznego stanu, a nie z wyboru najczęstszego wyniku.

**W swoich zadaniach zachowaj podział: wyprowadzenie → kod weryfikujący → wniosek.**

</div>

<div style="background-color:#f8f8f8;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## Przypomnienie techniczne — bramka MCX

---

MCX zmienia cel wtedy, gdy wszystkie kubity sterujące mają wartość 1. Poniższy przykład pokazuje składnię dla **trzech** sterowań; nie implementuje funkcji o zmiennej liczbie argumentów.

$$\lvert t\,x_2x_1x_0\rangle\mapsto
\lvert t\oplus x_0x_1x_2,\,x_2x_1x_0\rangle.$$

Wywołanie `qc.mcx([0, 1, 2], 3)` opisuje operację logiczną. Transpiler dobiera jej realizację dla symulatora lub urządzenia. Liczba bramek po dekompozycji nie musi być równa liczbie instrukcji wysokiego poziomu.

</div>

In [ ]:
qc_mcx_demo = prepare_basis_state("0111")
qc_mcx_demo.mcx([0, 1, 2], 3)
state_mcx_demo = Statevector.from_instruction(qc_mcx_demo)
pokaz_stan(state_mcx_demo, r"\psi_{\mathrm{MCX}}")
check_state(state_mcx_demo, Statevector.from_label("1111").data)
print(qc_mcx_demo.draw(output="text"))

<div style="background-color:#f4f0f9;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## Zadanie 1. Uniwersalna funkcja koniunkcji $n$-argumentowej

---

Napisz funkcję **`q_and(*args)`**, która przyjmuje dowolną liczbę argumentów $x_i\in\{0,1\}$ przy $n>1$, buduje obwód, wykonuje pomiar i zwraca $x_0\land x_1\land\ldots\land x_{n-1}$ jako `int`.

**A. Analitycznie:** zaproponuj mapowanie rejestru wejściowego i kubitów pomocniczych. Zapisz transformację na stanach bazowych, wyprowadź działanie wybranej konstrukcji oraz policz kubity. Dla co najmniej jednej pary przykładów z różnymi wynikami wypisz stan początkowy i przewidywany stan końcowy całego rejestru. Wyjaśnij, co pozostaje na kubitach roboczych.

**B. Python/Qiskit:** wybierz jedną metodę:

- **Kaskada Toffoliego:** wykorzystaj $n-1$ kubitów pomocniczych — $n-2$ robocze i jeden wynikowy — oraz bramki CCX.
- **MCX:** wykorzystaj $n$ kubitów wejściowych i jeden cel początkowo w $\lvert0\rangle$.

Zachowaj wejścia, nie twórz zbędnych rejestrów, waliduj liczbę i wartości argumentów. Uwzględnij limit zasobów symulatora. Przed pomiarem sprawdź wybrane pełne stany, następnie zmierz **tylko wynik** do `c0`. Pokaż tabelę testów funkcji i zliczenia dla wybranych przypadków.

**C. Wniosek:** omów poprawność, koszt zasobów i rolę kubitów pomocniczych. Jeżeli stosujesz kaskadę, wyjaśnij, czy i jak odtwarzasz robocze ancille do $\lvert0\rangle$.

**Przykłady oczekiwanego działania:** `q_and(1,1,1) → 1`, `q_and(1,0,1,1) → 0`.

</div>

<div style="background-color:#F6FFED;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

### Wskazówki do zadania 1

- Najpierw rozpatrz $n=2$, następnie $n=3$ i dopiero uogólnij indeksy. Dla $n=2$ kaskada sprowadza się do jednego CCX, bez roboczych ancilli.
- W metodzie MCX lista sterowań powinna zawierać indeksy wszystkich wejść; cel musi być innym kubitem.
- W kaskadzie kolejne cele zapisują koniunkcje prefiksów. Rozróżnij kubity robocze i cel końcowy.
- Przy $n-2$ kubitach roboczych i oddzielnym celu końcowym można po zapisaniu wyniku odwrócić **tylko obliczenia roboczych prefiksów**. Nie odwracaj operacji zapisującej wynik. Dla tej konstrukcji nie trzeba dodawać kolejnego kubitu do kopiowania wyniku.
- Logiczna liczba kubitów wynosi $2n-1$ dla opisanej kaskady albo $n+1$ dla MCX. Dodatkowe wymagania dekompozycji zależą od wybranej realizacji.
- Do prezentacji użyj `pokaz_stan`, `pokaz_prawdopodobienstwa(..., kubity=[cel])`, `pokaz_zliczenia` i `pokaz_tabele`.
- W testach nie ograniczaj się do samych jedynek: umieść pojedyncze zero na różnych pozycjach i sprawdź, czy wejścia pozostały bez zmian.

</div>

<div style="padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

 
### Moje obliczenia analityczne — zadanie 1

**Wybrana metoda i mapowanie kubitów:** …

**Transformacja oraz uzasadnienie dla ogólnego $n$:** …

**Liczba kubitów, limit zasobów, stan ancilli:** …

| Argumenty | Stan wejściowy całego rejestru | Przewidywany stan końcowy | Wynik |
|---|---|---|---|
| … | … | … | … |
| … | … | … | … |

In [ ]:
# TODO: zdefiniuj q_and(*args); zwracana wartość ma mieć typ int.
# TODO: odrzuć n <= 1 i niepoprawne wartości argumentów.
# Pomoc: bit(x) waliduje pojedynczy argument.
# TODO: oblicz liczbę kubitów i sprawdź limit przed utworzeniem obwodu.
# TODO: przygotuj wejścia, dodaj MCX albo własną kaskadę CCX.
# TODO: jeżeli czyścisz robocze ancille, zachowaj wynik końcowy.
# TODO: zmierz tylko kubit wyniku do c0 i uruchom Aer.
# Pomoc: counts = uruchom_pomiary(qc, shots=256).
# Pomoc: pokaz_zliczenia(counts); najczestszy_bit(counts).
# TODO: sprawdź, że poprawny idealny obwód daje tylko jeden wynik.
# Warto oddzielić konstrukcję obwodu od pomiaru, aby ułatwić testy stanu.

In [ ]:
# TODO: weryfikacja stanu PRZED dodaniem pomiarów dla wybranych wejść.
# TODO: expected1 zapisz na podstawie własnego rachunku, uwzględniając ancille.
# state1 = Statevector.from_instruction(qc1)
# pokaz_stan(state1, r"\psi_1")
# pokaz_stan(expected1, r"\psi_{1,\mathrm{teoria}}")
# check_state(state1, expected1)
# pokaz_prawdopodobienstwa(state1, kubity=[cel])
# TODO: sprawdź zachowanie wszystkich wejść i zadeklarowany stan roboczych ancilli.
# TODO: przetestuj q_and dla wszystkich wejść o długościach n=2,3,4.
# Pomoc: product((0,1), repeat=n); wzorzec klasyczny: int(all(args)).
# TODO: dodatkowo sprawdź same jedynki i pojedyncze zero na każdej pozycji dla n=5.
# TODO: sprawdź błędy dla q_and(), q_and(1), q_and(1,2), q_and(1,0.5).
# Pomoc: pokaz_tabele(["Argumenty","Oczekiwano","Uzyskano","Zgodność"], rows1).

<div style="padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">
 
### Moje wnioski — zadanie 1

**Czy rachunek, pełne stany i pomiary są zgodne?** …

**Co stało się z danymi wejściowymi i roboczymi ancillami?** …

**Jaki jest koszt wybranej metody i limit implementacji?** …

<div style="background-color:#f4f0f9;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## Zadanie 2. Pełny sumator kwantowy

---

Zbuduj **wielobitowy sumator kwantowy z propagacją przeniesienia** dla dwóch nieujemnych liczb całkowitych. Przygotuj funkcję **`build_add_circuit(a, b)`**, która zwraca obiekt `QuantumCircuit` z pomiarami, Natomiast **`quantum_add(a, b)` jest dostarczoną funkcją testującą**: uruchamia obwód na `AerSimulator` (`shots=1`) i zwraca wynik jako `int`. Nie modyfikuj jej podczas realizacji zadania.

**A. Analiza:**

1. Wyznacz wspólną szerokość wejść `n = max(1, a.bit_length(), b.bit_length())`. Uzasadnij, dlaczego wynik wymaga `n+1` bitów.
2. Zaplanuj rejestry $A$ i $B$ o długości $n$ oraz osobny, początkowo wyzerowany rejestr wyniku $C$ o długości $n+1$. Rozważ konstrukcję wykorzystującą **łącznie $3n+1$ kubitów**; jeśli potrzebujesz dodatkowych kubitów roboczych, wyjaśnij dlaczego.
3. Wyprowadź sumę i przeniesienie dla pojedynczej pozycji bitowej. Określ, kiedy przeniesienie jest jeszcze potrzebne, a kiedy dany kubit może pełnić inną rolę. Rejestry $A$ i $B$ mają zachować swoje wartości.
4. Dla $a=5$, $b=3$ prześledź kolejne bity, przeniesienia oraz przewidywaną etykietę pomiaru rejestru $C$.

**B. Implementacja:** wykorzystaj bramki `CX` i `CCX` poznane na laboratorium. Rozpocznij dodawanie od najmłodszych bitów (LSB), przekazując przeniesienie między kolejnymi stopniami. Nie obliczaj sumy klasycznie po to, by przygotować wynik bramkami `X`. Zachowaj obie liczby wejściowe, wykonaj pomiary rejestru $C$ i odczytaj cały ciąg wynikowy. Dodaj walidację argumentów oraz limit zasobów sprawdzany przed utworzeniem obwodu.

**C. Testy i wnioski:** uruchom dostarczony test tabelaryczny dla wszystkich 49 par `a,b` od 1 do 7. Następnie sprawdź małe przypadki przez `Statevector` **przed pomiarami** oraz przez symulator po pomiarach. Zweryfikuj wejścia $A$, $B$, wszystkie bity sumy i końcowe przeniesienie. Wskaż wpływ liczby kubitów na koszt symulacji.

**Uwagi:** dodatkowymi kubitami roboczymi można zarządzać na różne sposoby. Nie wymagamy ich oczyszczenia metodą *uncomputation*, lecz należy opisać ich stan końcowy, jeżeli występują. Samo zachowanie $A$ i $B$ oraz poprawnej sumy jest obowiązkowe.

**Przykłady:** `quantum_add(1, 0) == 1`, `quantum_add(3, 1) == 4`, `quantum_add(2, 3) == 5`.

</div>

<div style="background-color:#F6FFED;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

### Wskazówki do zadania 2 — od pełnego sumatora do dodawania wielobitowego

---

Wróć do pełnego sumatora z laboratorium. Dla pojedynczej pozycji bitowej oznacz wejścia przez $A_i$, $B_i$ i przeniesienie $P_i$, a wyjścia przez bit sumy $S_i$ i przeniesienie $P_{i+1}$.

$$S_i=A_i\oplus B_i\oplus P_i,$$
$$P_{i+1}=A_iB_i\oplus P_i(A_i\oplus B_i).$$

**Co należy samodzielnie zaprojektować?**

1. Zastanów się, dlaczego zwykłe przeniesienie dotychczasowego modułu na kolejne bity nie spełnia automatycznie wymagania zachowania **obu** rejestrów $A$ i $B$.
2. Bramki `CX` realizują XOR, a `CCX` pozwala warunkowo zmieniać kubit na podstawie dwóch wejść. Jak obliczyć oba składniki przeniesienia i nie utracić wartości $B_i$?
3. Najpierw wyznacz przeniesienie do **następnej** pozycji. Dopiero potem zastanów się, czy kubit zawierający **bieżące** przeniesienie jest nadal potrzebny. To może uprościć strukturę rejestrów.
4. Rozważ użycie pojedynczego rejestru $C$ długości $n+1$: podczas obliczeń jego kubity mogą zmieniać rolę. Zaplanuj, jakie informacje mają przechowywać **przed** i **po** danym stopniu.
5. Dla każdego stopnia sprawdź, czy po wykonaniu operacji rejestry $A$ i $B$ mają takie same wartości jak przed dodawaniem.
6. Przetestuj najpierw sumator 2-bitowy. W szczególności sprawdź `1+1`, `3+1` i przypadek bez przeniesienia.

**Pytania pomocnicze:** Czy każde przeniesienie musi mieć własny, osobny kubit aż do końca obliczeń? Jak wykorzystać odwracalność `CX` do przywrócenia wartości zmodyfikowanego chwilowo wejścia? Gdzie powinno znaleźć się końcowe przeniesienie?

Nie jest wymagane stosowanie konkretnego zestawu pięciu bramek ani tworzenie osobnej funkcji dla pojedynczego stopnia, lecz warto wyodrębnić powtarzalną sekwencję operacji i ją uzasadnić.

</div>

<div style="background-color:#F6FFED;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

### Wskazówki do zadania 2 — rejestry i przepływ informacji

---

- Przyjmij rejestry $A$ i $B$ o długości $n$ oraz rejestr $C$ długości $n+1$, początkowo w stanie $|0\ldots0\rangle$. Możesz wykorzystać obiekty `QuantumRegister` albo świadomie przypisane indeksy globalne.
- Gdy rejestry dodasz do obwodu w kolejności `(C, A, B)`, pełna etykieta stanu ma kolejność $|B\ A\ C\rangle$, a nie $|C\ B\ A\rangle$. Jeśli wybierzesz inną kolejność, odpowiednio zmień teorię i testy.
- Pomyśl o tym, że kubit $C_i$ może być najpierw nośnikiem przeniesienia wejściowego, a po jego wykorzystaniu — nośnikiem bitu sumy. **Nie zapisuj sumy za wcześnie**, jeśli bieżące przeniesienie jest jeszcze potrzebne do obliczenia kolejnego.
- Przetwarzaj bity od $i=0$ (LSB) do $i=n-1$ (MSB). Pierwsze przeniesienie ma wartość 0; wynik powinien być zapisany wyłącznie w $C$.
- Jeśli na chwilę zmodyfikujesz bit rejestru wejściowego w celu wyznaczenia XOR, zaplanuj przywrócenie jego oryginalnej wartości.
- Pokaż, z czego wynika liczba wykorzystanych kubitów. Wersja z $3n+1$ kubitami jest możliwa, ale uzasadnienie jej działania należy do rozwiązania zadania.
- Wariant z dodatkowymi kubitami jest dopuszczalny, jeśli wyjaśnisz ich użycie i sprawdzisz limit zasobów.

</div>

<div style="background-color:#F6FFED;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

### Wskazówki do zadania 2 — odczyt i testowanie

---

- Zmierz $C_i$ do klasycznego bitu $c_i$ dla $i=0,\ldots,n$. Gdy mierzysz tylko ten rejestr, klucz `get_counts()` ma postać $c_n\ldots c_1c_0$; do odczytu liczby użyj `int(key, 2)`.
- Wynik powinien mieć `n+1` bitów, także wtedy, gdy najstarszy bit wynosi 0: porównuj z `format(a+b, f"0{n+1}b")`.
- Dostarczona funkcja `quantum_add(a, b)` wykonuje **jeden strzał (`shots=1`)** i odczytuje `int(key, 2)`. Jest to wystarczające do podanego testu stanów bazowych na idealnym symulatorze. Dla dodatkowej kontroli deterministyczności możesz osobno uruchomić więcej strzałów i porównać cały rozkład zliczeń.

</div>

<div style="padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">
    
### Moje obliczenia analityczne — zadanie 2

**Wyprowadzenie pojedynczego stopnia oraz uzasadnienie odwracalności:** …

**Mapowanie rejestrów i liczba kubitów dla $5+3$:** …

| Pozycja $i$ | $A_i$ | $B_i$ | Przeniesienie wejściowe | $S_i$ | Przeniesienie wyjściowe |
|---|---|---|---|---|---|
| 0 | … | … | … | … | … |
| 1 | … | … | … | … | … |
| 2 | … | … | … | … | … |

**Końcowy rejestr $C$ i jego wartość dziesiętna:** …

**Pełny stan oczekiwany dla wybranego małego przykładu:** …

**Przypadek $a=b=0$, limit zasobów i polityka walidacji argumentów:** …

In [ ]:
# ZADANIE 2 — część do samodzielnego wykonania:
# Zaimplementuj wyłącznie funkcję budującą obwód.
# Funkcja testowa quantum_add oraz pętla poniżej są dostarczone.

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator


def build_add_circuit(a, b):
    # TODO 1: Sprawdź typy i nieujemność a, b. Ustal szerokość n,
    #         zapewniając co najmniej jeden bit dla przypadku (0, 0).
    # TODO 2: Zaplanuj rejestry A i B długości n oraz rejestr
    #         wyniku długości n+1. Czy wystarczy łącznie 3*n+1 kubitów?
    # TODO 3: Przed utworzeniem obwodu sprawdź założony limit kubitów.
    # TODO 4: Przygotuj A i B bramkami X; rejestr wyniku pozostaw
    #         początkowo w stanie |0...0>.
    # TODO 5: Zaprojektuj pojedynczy stopień pełnego sumatora,
    #         wykorzystując CX i CCX. Zachowaj wartości A[i] i B[i].
    #         W jakiej kolejności należy wyznaczyć przeniesienie i sumę?
    # TODO 6: Połącz stopnie od najmłodszego do najstarszego bitu.
    #         Zastanów się, czy kubit przeniesienia wejściowego
    #         można ponownie wykorzystać po zakończeniu stopnia.
    # TODO 7: Zmierz wszystkie n+1 kubitów wyniku do rejestru
    #         klasycznego o tej samej długości i zwróć QuantumCircuit.
    #         Pamiętaj o końcowym bicie przeniesienia.
    raise NotImplementedError("Uzupełnij build_add_circuit(a, b)")


# DOSTARCZONA FUNKCJA TESTOWA — nie wymaga modyfikacji.
def quantum_add(a, b):
    qc = build_add_circuit(a, b)

    simulator = AerSimulator()
    tqc = transpile(qc, backend=simulator)
    job = simulator.run(tqc, shots=1)
    job_status = job.result().get_counts()

    ll = list(job_status.keys())
    lv = int(ll[0], 2)

    return lv


# DOSTARCZONY TEST TABELARYCZNY — 49 par argumentów.
# TODO 8: Uruchom test po zaimplementowaniu build_add_circuit.
#         Porównaj wyniki z klasycznym a+b i zwróć uwagę
#         na przypadki z przeniesieniem do najstarszego bitu.
for b in range(1, 8):
    buf = []
    for a in range(1, 8):
        result = quantum_add(a, b)
        txt = "{v:2d}"
        buf.append(
            txt.format(v=a) + " + " + txt.format(v=b)
            + " =  " + txt.format(v=result)
        )
    print(buf)

<div style="padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

### Moje wnioski — zadanie 2

**W jaki sposób przeniesienie przechodzi pomiędzy stopniami i kiedy kubit można wykorzystać jako bit sumy?** …

**Czy rejestry A i B zachowują dane wejściowe? Jaka jest rola ostatniego bitu C?** …

**Jak rośnie liczba kubitów i koszt symulacji? Jaki limit zastosowano?** …

</div>

<div style="background-color:#fff9f0;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">
    
## Zadanie 3 (dodatkowe) — Czy superpozycja to tylko klasyczne losowanie?

---

W laboratorium poznaliśmy czterokubitowy, odwracalny sumator pełny. Dla klasycznych stanów wejściowych realizował on zwykłe dodawanie binarne.

Sprawdź, czym różni się działanie tego samego obwodu na **superpozycji stanów wejściowych** od jego działania na **klasycznie losowanych danych wejściowych**.

Przyjmij:

$$
A=1,\qquad B=0.
$$

Rozważ dwa przypadki przygotowania kubitu przeniesienia wejściowego $C_{\mathrm{in}}$:

- **Wariant A:** kubit w stanie $|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}$.
- **Wariant B:** klasyczne losowanie wartości $0$ lub $1$ z jednakowym prawdopodobieństwem.

### Polecenia

1. Wykorzystaj sumator pełny z laboratorium i wyznacz rozkład prawdopodobieństw wyników dodawania dla obu wariantów.

2. Bez wykonywania pomiaru po dodawaniu zastosuj obwód odwrotny $U^{-1}$, a następnie bramkę Hadamarda do kubitu $C_{\mathrm{in}}$.

3. Porównaj prawdopodobieństwa pomiaru kubitu $C_{\mathrm{in}}$ po całej sekwencji operacji. W wariancie B uwzględnij osobno oba klasycznie przygotowane stany i oblicz średni rozkład wyników.

4. Wyjaśnij, dlaczego oba warianty mogą dawać taki sam rozkład wyników dodawania, lecz różne wyniki końcowego eksperymentu.

5. Odpowiedz, jakie znaczenie mają spójność kwantowa, interferencja i operacje odwrotne (*uncomputation*) w bardziej złożonych algorytmach kwantowych.

### Wskazówki

- Obwód odwrotny można skonstruować, wykonując bramki sumatora w odwrotnej kolejności, albo wykorzystując metodę `inverse()` w Qiskit.
- Klasyczne losowanie nie jest tym samym co przygotowanie superpozycji. Trzeba je modelować jako mieszaninę statystyczną. **W wariancie B uśrednij prawdopodobieństwa** z dwóch osobnych symulacji z wagami po $1/2$; **nie sumuj amplitud** obu stanów, ponieważ otrzymałbyś superpozycję zamiast losowania.
- Zwróć uwagę na różnicę między prawdopodobieństwami pomiaru a amplitudami stanów kwantowych.
- Do weryfikacji wykorzystaj `Statevector` oraz funkcje pomocnicze poznane w laboratorium.

### Pytanie końcowe

**Dlaczego odwracalny sumator kwantowy może być przydatny w algorytmie kwantowym, nawet jeśli samo dodawanie nie jest szybsze niż na komputerze klasycznym?**

Zadanie jest nieobowiązkowe i przeznaczone dla osób zainteresowanych głębszym zrozumieniem właściwości obwodów kwantowych.


<div style="background-color:#F6FFED;color:#000000;padding:20px 25px;border-radius:12px;border:1px solid #000000;border-left-width:5px;margin:15px 0;box-shadow:0 2px 5px rgba(0,0,0,0.05);">

## Lista kontrolna przed oddaniem

- [ ] Uzupełniłem/am oba zadania: analizę, kod, testy i wnioski.
- [ ] Funkcje zadania 1 oraz `quantum_add(a,b)` wykonują obwód i zwracają `int`; `build_add_circuit(a,b)` zwraca obwód.
- [ ] Wartości oczekiwane obliczyłem/am niezależnie od symulacji.
- [ ] W zadaniu 1 wyjaśniłem/am liczbę i stan końcowy ancilli.
- [ ] W zadaniu 2 pokazałem/am rachunek 5+3 i końcowe przeniesienie.
- [ ] Zachowałem/am rejestry A i B, umieściłem/am wynik w C i opisałem/am ewentualne kubity robocze.
- [ ] Testowałem/am wybrane pełne stany bez pomiarów, nie tylko wyniki klasyczne.
- [ ] Testy obejmują małe pełne zestawy wejść, przypadki brzegowe i błędne argumenty.
- [ ] Mapowanie kubitów do bitów klasycznych zgadza się z odczytem `counts`.
- [ ] Sprawdziłem/am limit zasobów przed utworzeniem obwodu.
- [ ] Notebook można uruchomić od początku w świeżym kernelu i jest zapisany z wynikami.

</div>